# Word-level vs Sentence-level Accent Detection Analysis

This notebook compares the performance of accent detection at word-level vs sentence-level.

In [ ]:
import torch
import torchaudio
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, classification_report
import librosa

# Assuming our project modules are available
import sys
sys.path.append('..')

from config import LANGUAGES, SAMPLE_RATE
from features.extractor import FeatureExtractor
from models.classifiers import MFCCClassifier, HubertClassifier
from models.trainer import ModelTrainer

In [ ]:
def segment_audio_waveform(waveform, segment_type='sentence'):
    """
    Segment audio waveform into word-level or sentence-level chunks
    """
    if segment_type == 'word':
        # Simple segmentation - in practice, you would use a voice activity detector
        # or word alignment tool to get more accurate word segments
        segment_length = SAMPLE_RATE  # 1 second segments as approximation for words
    else:  # sentence
        segment_length = SAMPLE_RATE * 5  # 5 second segments for sentences
    
    segments = []
    for i in range(0, len(waveform), segment_length):
        segment = waveform[i:i+segment_length]
        # Pad if shorter than segment_length
        if len(segment) < segment_length:
            segment = torch.nn.functional.pad(segment, (0, segment_length - len(segment)))
        segments.append(segment)
    
    return segments

In [ ]:
def evaluate_segment_level(model, test_loader, segment_type, device='cpu'):
    """
    Evaluate model performance at specific segment level
    """
    model.eval()
    all_targets = []
    all_predictions = []
    
    with torch.no_grad():
        for data, targets in test_loader:
            data, targets = data.to(device), targets.to(device)
            outputs = model(data)
            _, predicted = torch.max(outputs, 1)
            
            all_targets.extend(targets.cpu().numpy())
            all_predictions.extend(predicted.cpu().numpy())
    
    accuracy = accuracy_score(all_targets, all_predictions)
    report = classification_report(all_targets, all_predictions, target_names=LANGUAGES, output_dict=True)
    
    print(f"{segment_type.capitalize()}-level Accuracy: {accuracy:.4f}")
    return accuracy, report

In [ ]:
# Placeholder for actual experiment
# In practice, you would:

"""
1. Load the dataset with word-level and sentence-level annotations
2. Create separate data loaders for each level
3. Train models on each level
4. Compare performance

# Example workflow:

# Load word-level dataset
# word_train_loader, word_val_loader, word_test_loader = load_segmented_data('word')

# Load sentence-level dataset
# sent_train_loader, sent_val_loader, sent_test_loader = load_segmented_data('sentence')

# Initialize models
# word_model = HubertClassifier(num_classes=len(LANGUAGES))
# sent_model = HubertClassifier(num_classes=len(LANGUAGES))

# Train models
# word_trainer = ModelTrainer(word_model)
# word_trainer.train(word_train_loader, word_val_loader)

# sent_trainer = ModelTrainer(sent_model)
# sent_trainer.train(sent_train_loader, sent_val_loader)

# Evaluate
# word_accuracy, word_report = evaluate_segment_level(word_model, word_test_loader, 'word')
# sent_accuracy, sent_report = evaluate_segment_level(sent_model, sent_test_loader, 'sentence')

# Compare results
# print(f"Word-level accuracy: {word_accuracy:.4f}")
# print(f"Sentence-level accuracy: {sent_accuracy:.4f}")

# Visualization
# levels = ['Word-level', 'Sentence-level']
# accuracies = [word_accuracy, sent_accuracy]

# plt.figure(figsize=(8, 6))
# bars = plt.bar(levels, accuracies, color=['skyblue', 'lightcoral'])
# plt.ylabel('Accuracy')
# plt.title('Word-level vs Sentence-level Accent Detection')
# plt.ylim(0, 1)

# Add value labels on bars
# for bar, acc in zip(bars, accuracies):
#     plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
#              f'{acc:.3f}', ha='center', va='bottom')

# plt.show()
"""

## Expected Results

Typically, sentence-level analysis provides more context for accent detection and may yield higher accuracy. However, word-level analysis might offer better interpretability and could be more robust to variations in speech length.

Key factors to consider:
- **Context**: Sentences provide more linguistic context
- **Variability**: Words may have more consistent pronunciation patterns
- **Robustness**: How well each approach generalizes to unseen data
- **Computational Efficiency**: Word-level processing may be faster